In [0]:
from pyspark.sql import functions as F

# =========================
# Read Silver tables
# =========================
patient = spark.table("silver_patient")
provider = spark.table("silver_provider")
dept = spark.table("silver_department")
visit = spark.table("silver_er_visit")
followup = spark.table("silver_followup")

# =========================
# 1. Dimension Tables
# =========================

dim_patient = (
    patient
    .select(
        "patient_id",
        "gender_cd",
        "age_band_raw",
        "insurance_plan",
        "lang_pref",
        "chronic_cnt",
        "risk_ind",
        "zip3"
    )
    .dropDuplicates(["patient_id"])
)

dim_provider = (
    provider
    .select(
        "provider_id",
        "provider_role_raw",
        "exp_yrs",
        "shift_code",
        "active_ind"
    )
    .dropDuplicates(["provider_id"])
)

dim_department = (
    dept
    .select(
        "dept_code",
        "dept_name",
        "hospital_name",
        "city",
        "state"
    )
    .dropDuplicates(["dept_code"])
)

# =========================
# 2. Fact ER Visit
# =========================

fact_er_visit = (
    visit
    # Raw time differences
    .withColumn("wait_to_triage_min_raw", (F.col("triage_ts").cast("long") - F.col("arrival_ts").cast("long")) / 60.0)
    .withColumn("wait_to_bed_min_raw", (F.col("bed_ts").cast("long") - F.col("arrival_ts").cast("long")) / 60.0)
    .withColumn("wait_to_provider_min_raw", (F.col("md_seen_ts").cast("long") - F.col("arrival_ts").cast("long")) / 60.0)
    .withColumn("los_min_raw", (F.col("discharge_ts").cast("long") - F.col("arrival_ts").cast("long")) / 60.0)

    # Business safe fallbacks
    .withColumn(
        "wait_to_triage_min",
        F.when(F.col("wait_to_triage_min_raw").isNotNull(), F.col("wait_to_triage_min_raw"))
         .when(F.col("wait_to_bed_min_raw").isNotNull(), F.col("wait_to_bed_min_raw"))
         .when(F.col("wait_to_provider_min_raw").isNotNull(), F.col("wait_to_provider_min_raw"))
         .otherwise(None)
    )
    .withColumn(
        "wait_to_bed_min",
        F.when(F.col("wait_to_bed_min_raw").isNotNull(), F.col("wait_to_bed_min_raw"))
         .when(F.col("wait_to_provider_min_raw").isNotNull(), F.col("wait_to_provider_min_raw"))
         .otherwise(None)
    )
    .withColumn(
        "wait_to_provider_min",
        F.when(F.col("wait_to_provider_min_raw").isNotNull(), F.col("wait_to_provider_min_raw"))
         .when(F.col("wait_to_bed_min_raw").isNotNull(), F.col("wait_to_bed_min_raw"))
         .otherwise(None)
    )
    .withColumn(
        "los_min",
        F.when(F.col("los_min_raw").isNotNull(), F.col("los_min_raw"))
         .otherwise(None)
    )

    # Clamp clearly invalid negative values to null
    .withColumn("wait_to_triage_min", F.when(F.col("wait_to_triage_min") < 0, None).otherwise(F.col("wait_to_triage_min")))
    .withColumn("wait_to_bed_min", F.when(F.col("wait_to_bed_min") < 0, None).otherwise(F.col("wait_to_bed_min")))
    .withColumn("wait_to_provider_min", F.when(F.col("wait_to_provider_min") < 0, None).otherwise(F.col("wait_to_provider_min")))
    .withColumn("los_min", F.when(F.col("los_min") < 0, None).otherwise(F.col("los_min")))

    # Date and time features
    .withColumn("arrival_hour", F.hour("arrival_ts"))
    .withColumn("arrival_day_of_week", F.date_format("arrival_ts", "E"))
    .withColumn("arrival_month", F.month("arrival_ts"))
    .withColumn("arrival_date", F.to_date("arrival_ts"))
    .withColumn("is_weekend", F.when(F.dayofweek("arrival_ts").isin([1, 7]), 1).otherwise(0))
    .withColumn("is_peak_hour", F.when((F.col("arrival_hour") >= 10) & (F.col("arrival_hour") <= 21), 1).otherwise(0))

    # Quality flags
    .withColumn("invalid_bed_before_arrival", F.when(F.col("bed_ts") < F.col("arrival_ts"), 1).otherwise(0))
    .withColumn("invalid_provider_before_arrival", F.when(F.col("md_seen_ts") < F.col("arrival_ts"), 1).otherwise(0))
    .withColumn("invalid_discharge_before_arrival", F.when(F.col("discharge_ts") < F.col("arrival_ts"), 1).otherwise(0))

    # Targets
    .withColumn(
        "delay_flag",
        F.when(F.col("wait_to_provider_min").isNull(), 0)
         .when(F.col("wait_to_provider_min") > 30, 1)
         .otherwise(0)
    )
)

# =========================
# 3. Fact Followup
# =========================

fact_followup = (
    followup
    .withColumn(
        "followup_risk_flag",
        F.when(
            (F.col("followup_needed_ind") == 1) & (F.col("completed_ind") == 0),
            1
        ).otherwise(0)
    )
)

# =========================
# 4. Feature Table for Modeling
# =========================

feature_er_visit_model = (
    fact_er_visit.alias("v")
    .join(dim_patient.alias("p"), on="patient_id", how="left")
    .join(dim_provider.alias("pr"), on="provider_id", how="left")
    .join(
        fact_followup.select("visit_id", "followup_needed_ind", "followup_type", "rec_followup_days", "completed_ind", "followup_risk_flag").alias("f"),
        on="visit_id",
        how="left"
    )
    .fillna({
        "followup_needed_ind": 0,
        "completed_ind": 0,
        "followup_risk_flag": 0
    })
)

# =========================
# 5. Daily Aggregate Table
# =========================

agg_er_daily_metrics = (
    fact_er_visit
    .groupBy("arrival_date")
    .agg(
        F.count("*").alias("total_visits"),
        F.avg("wait_to_triage_min").alias("avg_wait_to_triage_min"),
        F.avg("wait_to_bed_min").alias("avg_wait_to_bed_min"),
        F.avg("wait_to_provider_min").alias("avg_wait_to_provider_min"),
        F.avg("los_min").alias("avg_los_min"),
        F.sum("delay_flag").alias("total_delays"),
        F.avg("delay_flag").alias("delay_rate"),
        F.sum("lwbs_ind").alias("lwbs_count"),
        F.avg("lwbs_ind").alias("lwbs_rate"),
        F.sum("admit_ind").alias("admit_count"),
        F.avg("admit_ind").alias("admit_rate"),
        F.sum("obs_ind").alias("observation_count"),
        F.avg("obs_ind").alias("observation_rate")
    )
)

# =========================
# 6. Write Gold Tables
# =========================

dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_patient")

dim_provider.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_provider")

dim_department.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_department")

fact_er_visit.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_er_visit")

fact_followup.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_followup")

feature_er_visit_model.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("feature_er_visit_model")

agg_er_daily_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("agg_er_daily_metrics")
# =========================
# 7. Validation Checks
# =========================

print("dim_patient")
spark.sql("SELECT COUNT(*) AS row_count FROM dim_patient").show()

print("dim_provider")
spark.sql("SELECT COUNT(*) AS row_count FROM dim_provider").show()

print("dim_department")
spark.sql("SELECT COUNT(*) AS row_count FROM dim_department").show()

print("fact_er_visit")
spark.sql("SELECT COUNT(*) AS row_count FROM fact_er_visit").show()

print("fact_followup")
spark.sql("SELECT COUNT(*) AS row_count FROM fact_followup").show()

print("feature_er_visit_model")
spark.sql("SELECT COUNT(*) AS row_count FROM feature_er_visit_model").show()

print("agg_er_daily_metrics")
spark.sql("SELECT COUNT(*) AS row_count FROM agg_er_daily_metrics").show()

print("visit duplicate check")
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT visit_id) AS distinct_visits
FROM fact_er_visit
""").show()

print("null check for key metrics")
spark.sql("""
SELECT
    SUM(CASE WHEN triage_ts IS NULL THEN 1 ELSE 0 END) AS triage_ts_nulls,
    SUM(CASE WHEN bed_ts IS NULL THEN 1 ELSE 0 END) AS bed_ts_nulls,
    SUM(CASE WHEN md_seen_ts IS NULL THEN 1 ELSE 0 END) AS md_seen_ts_nulls,
    SUM(CASE WHEN discharge_ts IS NULL THEN 1 ELSE 0 END) AS discharge_ts_nulls,
    SUM(CASE WHEN wait_to_triage_min IS NULL THEN 1 ELSE 0 END) AS wait_to_triage_nulls,
    SUM(CASE WHEN wait_to_bed_min IS NULL THEN 1 ELSE 0 END) AS wait_to_bed_nulls,
    SUM(CASE WHEN wait_to_provider_min IS NULL THEN 1 ELSE 0 END) AS wait_to_provider_nulls,
    SUM(CASE WHEN los_min IS NULL THEN 1 ELSE 0 END) AS los_nulls
FROM fact_er_visit
""").show()

print("overall KPI check")
spark.sql("""
SELECT
    AVG(wait_to_provider_min) AS avg_wait_to_provider_min,
    AVG(los_min) AS avg_los_min,
    AVG(delay_flag) AS delay_rate,
    AVG(lwbs_ind) AS lwbs_rate,
    AVG(admit_ind) AS admit_rate
FROM fact_er_visit
""").show()

print("sample fact rows")
spark.sql("""
SELECT
    visit_id,
    patient_id,
    provider_id,
    dept_code,
    arrival_ts,
    triage_ts,
    bed_ts,
    md_seen_ts,
    discharge_ts,
    arrival_mode,
    acuity_cd,
    chief_complaint_raw,
    disposition,
    visit_status,
    wait_to_triage_min,
    wait_to_bed_min,
    wait_to_provider_min,
    los_min,
    delay_flag
FROM fact_er_visit
LIMIT 10
""").show(truncate=False)

dim_patient
+---------+
|row_count|
+---------+
|    16500|
+---------+

dim_provider
+---------+
|row_count|
+---------+
|       58|
+---------+

dim_department
+---------+
|row_count|
+---------+
|        1|
+---------+

fact_er_visit
+---------+
|row_count|
+---------+
|    20966|
+---------+

fact_followup
+---------+
|row_count|
+---------+
|     9336|
+---------+

feature_er_visit_model
+---------+
|row_count|
+---------+
|    20966|
+---------+

agg_er_daily_metrics
+---------+
|row_count|
+---------+
|      547|
+---------+

visit duplicate check
+----------+---------------+
|total_rows|distinct_visits|
+----------+---------------+
|     20966|          20966|
+----------+---------------+

null check for key metrics
+---------------+------------+----------------+------------------+--------------------+-----------------+----------------------+---------+
|triage_ts_nulls|bed_ts_nulls|md_seen_ts_nulls|discharge_ts_nulls|wait_to_triage_nulls|wait_to_bed_nulls|wait_to_provider_nulls